# 采购增量 数据清洗 — 逐项确认

全部排查一次性生成，**确认一项过一项**。每项统一三段式：**总结 → 明细 → 建议**，结论写在「你的结论」。
数据只读 raw、内存预演；确认过的规则第四步固化进 processed，raw 永不动。窗口 2025-10 ~ 2026-09。

In [1]:
import sys; sys.path.insert(0, "src")
import pandas as pd, numpy as np
pd.set_option("display.max_columns", 30, "display.width", 200)
from read_data import read_inbound

df = read_inbound("2025-10-01", "2026-09-30")
df["出入库日期"] = pd.to_datetime(df["出入库日期"])
TOTAL_ROWS, TOTAL_AMT = len(df), df["采购金额(万)"].sum()

# ===== 项一 全列体检（已确认） =====
profile = pd.DataFrame({"类型": df.dtypes.astype(str), "非空率": df.notna().mean().round(3),
                        "取值种数": df.nunique()}).sort_values("非空率")
profile.head(8)

,类型,非空率,取值种数
购退单价,float64,0.003,470
多组织名称,object,0.048,1
货号,object,0.849,16315
财务内部外供应商类型,str,0.977,3
采购类型名称,object,0.988,9
采购单价,float64,0.997,9640
生产地址,str,0.998,2687
生产国,str,0.999,31


### 项一 · 全列体检

**总结**：356,164 行 / 51 列，不含税金额 12.04 亿，12 个月无断档。缺失只在 4 列：`购退单价`(99.7%空，仅退货填，正常)、`财务内部外供应商类型`(2.3%空→项三)、`采购类型名称`(1.2%)、`货号`(15%)——均非聚合必需。业务关键列 100% 非空。
**建议**：无需处理。**你的结论**：✅ 已确认（2026-11-02）。

---

### 项二 · 完全重复行

**总结**：2,575 行（0.72%）/ 816 万整行重复，12 个月均匀分布；组内批号全部一致=真导出重复；与 nc/ebs 双系统无关（0 单据跨系统）。**不剔会把单据 3084000080-1（智能采血系统，11 行/190 万）的采购量虚增 10 倍**。
**建议**：剔除（每组留 1 行，拟剔 1,482 行），仅批号不同的 3,792 行合法拆行保留。
**你的结论**：⏸️ 待业务确认，暂不剔除（2026-11-02）。

In [2]:
# ===== 项二 完全重复行（待确认） =====
dup_mask = df.duplicated(keep=False)
hit = df.loc[dup_mask]
grp_cols = list(df.columns)
batch_varied = sum(1 for _, s in hit.groupby(grp_cols, dropna=False) if s["批次号"].nunique(dropna=False) > 1)
key = ["单据编码","物料编码","数量","采购金额(万)","出入库日期"]
hk = df.loc[df.duplicated(subset=key, keep=False)]
legal_split = int(hk.groupby(key, dropna=False)["批次号"].transform("nunique").gt(1).sum())
display(pd.DataFrame([
    {"项":"整行重复","结果":f"{int(dup_mask.sum()):,} 行 / {hit['采购金额(万)'].sum():,.1f} 万"},
    {"项":"组内批号不一致","结果":f"{batch_varied} 组 → 全部一致，真重复"},
    {"项":"仅批号不同的合法拆行","结果":f"{legal_split:,} 行 → 保留"},
    {"项":"拟剔除（每组留1行）","结果":f"{int(df.duplicated().sum()):,} 行"}]))

g = (hit.groupby(["单据编码","物料编码","出入库日期"]).agg(组内行数=("数量","size"),
      合计金额万=("采购金额(万)","sum"), 物料名称=("物料名称","first"),
      供应商名称=("供应商名称","first")).reset_index())
g["_m"] = g["出入库日期"].dt.strftime("%Y-%m")
display(g.sort_values("合计金额万", ascending=False).head(5))
print(f"自查定位：源表按整行全列一致筛，{len(g)} 组/{len(hit):,} 行，单据 {g['单据编码'].nunique()} 个")

,项,结果
0,整行重复,"2,575 行 / 816.1 万"
1,组内批号不一致,0 组 → 全部一致，真重复
2,仅批号不同的合法拆行,"3,792 行 → 保留"
3,拟剔除（每组留1行）,"1,482 行"


,单据编码,物料编码,出入库日期,组内行数,合计金额万,物料名称,供应商名称,_m
801,3084000080 - 1,12021592,2026-06-23,11,189.999997,智能采血系统(智能备管机),重庆微标惠智医疗信息技术有限公司,2026-06
819,CR2001260400341,21000062,2026-04-03,4,52.800000,人乳头瘤病毒(HPV)核酸分型检测试剂盒,杭州迪安生物技术有限公司,2026-04
929,CR2015260700169,11038405,2026-07-28,4,33.750000,"胎儿染色体非整倍体(T21,T18,T13)检测试剂盒-试剂盒 A(1/2)",云南盛时迪安生物科技有限公司,2026-07
920,CR2015251000111,11044389,2025-10-28,2,24.373440,肺炎支原体核酸及耐药突变位点检测试剂盒,云南盛时迪安生物科技有限公司,2025-10
217,30020033803 - 1,11043871,2026-09-09,2,11.750400,25-羟基维生素 D 检测试剂盒25-Hydroxy Vitamin DS EIA（）,南京优阳生物科技有限公司,2026-09


自查定位：源表按整行全列一致筛，1068 组/2,575 行，单据 1011 个


### 项三 · 关键字段空值（供应商类型）

**总结**：唯一有空值的关键列是供应商类型——8,118 行 / 1,809 万（1.5%），12 个月均匀；其余关键列零空值。缺失行在「自定义内外部供应商」列均为外部供应商。
**建议**：清洗时填充「外部单位」（该列标准名定为**供应商类型**），不拿自定义列补标防口径混用。
**你的结论**：✅ 已确认：填外部（2026-11-02）。

In [3]:
# ===== 项三 空值（已确认：填外部） =====
keys = ["物料编码","数量","采购金额(万)","出入库日期","板块","供应商名称","财务内部外供应商类型"]
display(pd.DataFrame({"空值行数":[int(df[c].isna().sum()) for c in keys],
    "占比":[f"{df[c].isna().mean():.2%}" for c in keys],
    "金额(万)":[round(float(df.loc[df[c].isna(),"采购金额(万)"].sum()),1) for c in keys]},
    index=keys).sort_values("空值行数", ascending=False).head(3))
m = df["财务内部外供应商类型"].isna()
top = df.loc[m].groupby("供应商名称").agg(缺失行数=("数量","size"), 金额万=("采购金额(万)","sum")).round(1)
top["自定义列"] = df.loc[m].groupby("供应商名称")["自定义内外部供应商（凯莱谱、生物为外部）"].agg(lambda s: s.mode().iat[0])
display(top.sort_values("缺失行数", ascending=False).head(3))

,空值行数,占比,金额(万)
财务内部外供应商类型,8118,2.28%,2044.2
物料编码,0,0.00%,0.0
数量,0,0.00%,0.0


,缺失行数,金额万,自定义列
供应商名称,,,
聊城市鑫科金诺医疗器械有限公司,3609,235.4,外部供应商
杭州济亢医疗器械有限公司,1590,540.7,外部供应商
广东金发科技有限公司,1018,148.8,外部供应商


**建议**：格式修复——全部文本列 strip + 去 \r\n\t（只动格式不改业务值），留痕。
**你的结论**：✅ 已确认（2026-11-02）。

In [4]:
# ===== 项四 有效性（待确认） =====
enum_cols = ["分析类型","出入库类型名称","板块","财务内部外供应商类型","是否多组织内交易","数据来源系统"]
rows = [{"列":c, "取值":repr(v), "行数":f"{n:,}", "占比":f"{n/len(df):.1%}"}
        for c in enum_cols for v, n in df[c].value_counts(dropna=False).items()]
display(pd.DataFrame(rows))
by_sys = df.groupby("数据来源系统")["公司编码"].apply(set)
display(pd.DataFrame([{"项":"nc公司数","结果":len(by_sys.get("nc",set()))},
    {"项":"ebs公司数","结果":len(by_sys.get("ebs",set()))},
    {"项":"两系统重叠公司","结果":len(by_sys.get("nc",set()) & by_sys.get("ebs",set()))},
    {"项":"窗口外/未来日期","结果":f"{int(((df['出入库日期']<'2025-10-01')|(df['出入库日期']>'2026-09-30')).sum())} / {int((df['出入库日期']>pd.Timestamp.today()).sum())} 行"}]))
dirty = {c:int(df[c].dropna().astype(str).str.contains(r"\r|\n|_x000d_|^\s|\s$", regex=True).sum())
         for c in df.select_dtypes(include=["object","str"]).columns}
display(pd.DataFrame([{"列":k,"脏字符行数":v} for k,v in sorted(dirty.items(), key=lambda x:-x[1]) if v]))

,列,取值,行数,占比
0,分析类型,'入库',"356,164",100.0%
1,出入库类型名称,'普通采购入库',"181,878",51.1%
2,出入库类型名称,'PO接收',"172,698",48.5%
3,出入库类型名称,'退回供应商',998,0.3%
4,出入库类型名称,'采购退货',590,0.2%
5,板块,'检验产品',"174,121",48.9%
6,板块,'检验',"143,712",40.3%
7,板块,'诊断',"38,331",10.8%
8,财务内部外供应商类型,'外部单位',"250,230",70.3%
9,财务内部外供应商类型,'内部关联单位',"96,224",27.0%


,项,结果
0,nc公司数,34
1,ebs公司数,18
2,两系统重叠公司,0
3,窗口外/未来日期,0 / 0 行


,列,脏字符行数
0,生产地址,88
1,货号,87
2,规格,37
3,物料名称,17
4,批次号,10


**你的结论**：✅ 已确认：保留不改，聚合一律用金额列（2026-11-02）。

In [5]:
# ===== 项五 一致性勾稽（待确认） =====
nz = df[df["数量"]!=0]
chk = pd.DataFrame({
    "含税": (nz["采购单价"]*nz["数量"]/1e4 - nz["采购金额(万)"]).abs() / nz["采购金额(万)"].replace(0,np.nan),
    "不含税": (nz["采购不含税单价"]*nz["数量"]/1e4 - nz["采购不含税金额(万)"]).abs() / nz["采购不含税金额(万)"].replace(0,np.nan)})
bad = (chk > 0.01).any(axis=1)
bad_rows = df.loc[nz.index[bad.fillna(False)]].copy()
bad_rows["偏差%"] = (chk.max(axis=1)[bad.fillna(False)]*100).round(1)
display(pd.DataFrame([{"项":"单价×数量 vs 金额 偏差>1%","结果":f"{len(bad_rows):,} 行 / {bad_rows['采购不含税金额(万)'].sum():,.1f} 万"}]))

ym_bad = df[(df["年份"]!=df["出入库日期"].dt.year) | (df["月份"]!=df["出入库日期"].dt.month)]
display(pd.DataFrame([{"项":"年份月份 vs 出入库日期 不一致","结果":f"{len(ym_bad):,} 行"}]))

cols = ["单据编码","物料编码","出入库日期","数量","采购单价","采购金额(万)","采购不含税单价","采购不含税金额(万)","偏差%"]
display(bad_rows.sort_values("偏差%", ascending=False)[cols].head(5))
print(f"自查定位：偏差>1% 的单据号：{bad_rows['单据编码'].unique()[:5].tolist()} 等 {bad_rows['单据编码'].nunique()} 个")

,项,结果
0,单价×数量 vs 金额 偏差>1%,7 行 / 8.1 万


,项,结果
0,年份月份 vs 出入库日期 不一致,0 行


,单据编码,物料编码,出入库日期,数量,采购单价,采购金额(万),采购不含税单价,采购不含税金额(万),偏差%
138755,CR2003260300036,11003300,2026-03-03,10.0,288.00,0.14400,254.867257,0.127434,100.0
59575,CR2017251200030,11035125,2025-12-04,1.0,297.50,0.08925,263.274336,0.078982,66.7
1750,CR2001251000145,11017827,2025-10-01,70.0,618.83,8.66362,547.637168,7.666920,50.0
200923,CR2001260500485,11027311,2026-05-07,3.0,15.00,0.00300,13.274336,0.002655,50.0
200663,CR2001260500485,11027307,2026-05-07,2.0,450.00,0.13500,398.230088,0.119469,33.3


自查定位：偏差>1% 的单据号：['CR2001251000145', 'CR2017251200030', 'CR2007260200020', 'CR2003260300036', 'CR2001260500485'] 等 6 个


**你的结论**：✅ 已确认：全部保留，报告加「免费数量占比」参考列（2026-11-02）。

In [6]:
# ===== 项六 准确性（待确认） =====
neg = df[df["数量"]<0]
free = df[(df["数量"]!=0) & (df["采购金额(万)"]==0)]
negamt = df[df["采购金额(万)"]<0]
display(pd.DataFrame([
    {"项":"负数量（退货冲减）","行数":f"{len(neg):,}", "数量":f"{neg['数量'].sum():,.0f}", "金额(万)":f"{neg['采购不含税金额(万)'].sum():,.1f}"},
    {"项":"有数量无金额（免费配套）","行数":f"{len(free):,}", "数量":f"{free['数量'].sum():,.0f}", "金额(万)":"0（按口径计0）"},
    {"项":"负金额","行数":f"{len(negamt):,}", "数量":"—", "金额(万)":f"{negamt['采购不含税金额(万)'].sum():,.1f}"}]))
display(neg.groupby("出入库类型名称").agg(行数=("数量","size"), 数量=("数量","sum")).round(0))
display(free.groupby("物料名称").agg(行数=("数量","size"), 免费数量=("数量","sum")).sort_values("免费数量",ascending=False).head(5))
up = df.nlargest(5, "采购单价")[["单据编码","物料名称","数量","采购单价","出入库日期"]]
display(up.assign(**{"含税单价": up["采购单价"].round(0)}).drop(columns="采购单价"))

,项,行数,数量,金额(万)
0,负数量（退货冲减）,"1,629","-36,685",-879.0
1,有数量无金额（免费配套）,"40,492","5,150,223",0（按口径计0）
2,负金额,"1,465",—,-879.0


,行数,数量
出入库类型名称,,
普通采购入库,41,-399.0
退回供应商,998,-12621.0
采购退货,590,-23665.0


,行数,免费数量
物料名称,,
吸管,165,1412293.0
反应杯,220,873506.0
一次性使用宫颈采样拭子,393,762131.0
细胞保存液,571,679201.0
一次性采样拭子,128,418686.8


,单据编码,物料名称,数量,出入库日期,含税单价
257040,3070005451 - 1,医用血管造影 X 射线机,1.0,2026-07-03,13037000.0
313023,3072006033 - 1,飞行时间质谱检测系统,1.0,2026-08-28,2038760.0
138745,3065000808 - 1,基因分析仪 Genetic Analyzer,1.0,2026-03-03,2030000.0
124538,CR1001260200273,液相色谱串联质谱检测系统-三重四级杆串联质谱仪,1.0,2026-02-03,1750000.0
281921,3071001145 - 1,液相色谱串联质谱检测系统,1.0,2026-07-30,1700000.0


In [7]:
# 项六补充 · 退货单价填充核查（字段确认新增）
ret = df[df["出入库类型名称"] == "退回供应商"]
display(pd.DataFrame([
    {"项": "退回供应商行数", "结果": f"{len(ret):,}"},
    {"项": "其中含税单价为空", "结果": f"{int(ret['采购单价'].isna().sum()):,}"},
    {"项": "其中购退单价有值", "结果": f"{int(ret['购退单价'].notna().sum()):,}"},
    {"项": "清洗后可补齐", "结果": "是" if (ret["采购单价"].isna() & ret["购退单价"].notna()).sum() == ret["采购单价"].isna().sum() else "部分"}]))

,项,结果
0,退回供应商行数,998
1,其中含税单价为空,998
2,其中购退单价有值,998
3,清洗后可补齐,是


**你的结论**：✅ 已确认：无需处理（2026-11-02）。

In [8]:
# ===== 项七 及时性（待确认） =====
monthly = df.groupby(df["出入库日期"].dt.strftime("%Y-%m")).agg(行数=("数量","size"), 金额万=("采购金额(万)","sum")).round(1)
display(monthly.T)
alldays = pd.date_range("2025-10-01","2026-09-30",freq="D")
missing = alldays.difference(df["出入库日期"].unique())
display(pd.DataFrame([{"项":"有数据的自然日","结果":f"{df['出入库日期'].nunique()} / {len(alldays)}"},
    {"项":"无记录日期","结果":f"{len(missing)} 天" + (f"：{[str(d.date()) for d in missing[:8]]}" if len(missing)<=8 else "")}]))

出入库日期,2025-10,2025-11,2025-12,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06,2026-07,2026-08,2026-09
行数,23082.0,29862.0,32051.0,33198.0,18397.0,27753.0,31853.0,26286.0,29458.0,30759.0,31616.0,41849.0
金额万,8831.3,12169.6,13074.6,11800.8,7565.2,10937.5,10793.2,9506.8,11762.2,12871.4,11240.1,15374.0


,项,结果
0,有数据的自然日,343 / 365
1,无记录日期,22 天


---

## 总账：套用已确认处置，整体跑一遍

已确认处置：① 文本脏字符 strip（项四）；② 供应商类型空值填外部（项三）；③ 含税单价空值用购退单价补（字段确认）。
项二重复行**待业务确认，暂不剔除**——总账暂无剔除项，项二定夺后重跑本节。

In [9]:
clean = df.copy()
n_text = sum(int(clean[c].dropna().astype(str).str.contains(r"\r|\n|_x000d_|^\s|\s$", regex=True).sum())
             for c in clean.select_dtypes(include=["object","str"]).columns)
for c in clean.select_dtypes(include=["object","str"]).columns:
    clean[c] = clean[c].astype("string").str.replace(r"[\r\n\t]", "", regex=True).str.strip()
n_type = int(clean["财务内部外供应商类型"].isna().sum())
clean["财务内部外供应商类型"] = clean["财务内部外供应商类型"].fillna("外部单位")
n_price = int(clean["采购单价"].isna().sum())
clean["采购单价"] = clean["采购单价"].fillna(clean["购退单价"])

# 采购模式归一（格式，不减行）：去掉"合同"后缀，统一为 短名
mode_map = {"定向采购合同": "定向采购", "集中采购合同": "集中采购", "指定采购合同": "指定采购",
            "自行采购合同": "自行采购", "普通采购合同": "其他"}
n_mode = int(clean["采购类型名称"].isin(mode_map).sum())
clean["采购模式"] = clean["采购类型名称"].map(lambda x: mode_map.get(x, x)).fillna("未标注")

audit = pd.DataFrame([
    {"项": "清洗前", "行数": f"{len(df):,}", "含税金额(万)": f"{df['采购金额(万)'].sum():,.1f}"},
    {"项": "文本脏字符修复（格式，不减行）", "行数": f"{n_text}", "含税金额(万)": "—"},
    {"项": "供应商类型填充外部", "行数": f"{n_type:,}", "含税金额(万)": "—"},
    {"项": "含税单价补购退单价", "行数": f"{n_price:,}", "含税金额(万)": "—"},
    {"项": "采购模式归一（格式，不减行）", "行数": f"{n_mode:,}", "含税金额(万)": "—"},
    {"项": "剔除（项二挂起，暂无）", "行数": "0", "含税金额(万)": "—"},
    {"项": "清洗后", "行数": f"{len(clean):,}", "含税金额(万)": f"{clean['采购金额(万)'].sum():,.1f}"},
])
display(audit)
print("金额守恒:", abs(clean['采购金额(万)'].sum() - df['采购金额(万)'].sum()) < 1e-6, " 行数守恒:", len(clean) == len(df))

,项,行数,含税金额(万)
0,清洗前,"356,164","135,926.6"
1,文本脏字符修复（格式，不减行）,239,—
2,供应商类型填充外部,"8,118",—
3,含税单价补购退单价,998,—
4,采购模式归一（格式，不减行）,"179,714",—
5,剔除（项二挂起，暂无）,0,—
6,清洗后,"356,164","135,926.6"


金额守恒: True  行数守恒: True


---

## 附：查看清洗后的完整数据（可选运行）

下面一格输出清洗后底表的预览：行数、27 个标准字段、前 10 行样例。确认无误后，这份明细就是 processed 层要落盘的数据，也是第四步指标计算的输入。

In [10]:
# 清洗后完整数据（可选运行）
# 全量渲染 35 万行会明显卡顿，所以默认走导出；想在本页全量看，把 SHOW_FULL 改为 True 再运行。
SHOW_FULL = False
preview_cols = ["出入库日期","物料编码","物料名称","规格","主单位","一级分类","二级分类","三级分类","四级分类","五级分类",
                "品牌","主品牌","货号","供应商名称","板块","公司编码","公司名称","公司简称","大区","数据来源系统",
                "设备试剂分类","采购模式","数量","采购单价","采购金额(万)","税率","财务内部外供应商类型"]
out = clean[preview_cols]
if SHOW_FULL:
    with pd.option_context("display.max_rows", None, "display.max_columns", None):
        display(out)
else:
    path = "outputs/清洗后明细_全量.xlsx"
    out.to_excel(path, index=False)
    print(f"清洗后明细全量 {len(out):,} 行 × {len(preview_cols)} 个标准字段")
    print(f"已导出：{path}（Excel 可直接打开翻全量）")
    print("想在本页逐行看：把本格 SHOW_FULL 改为 True 再运行。")

清洗后明细全量 356,164 行 × 27 个标准字段
已导出：outputs/清洗后明细_全量.xlsx（Excel 可直接打开翻全量）
想在本页逐行看：把本格 SHOW_FULL 改为 True 再运行。
